# Complex Workflows — Interactive Comparison

This notebook demonstrates Chain of Draft (CoD), System 2 Attention (S2A), Prompt Chaining,
and Meta Prompting by comparing their outputs on identical problems. Run top-to-bottom to see the differences.

**Techniques covered:**
1. **Chain of Draft (CoD)** — concise reasoning steps vs. verbose Chain-of-Thought
2. **System 2 Attention (S2A)** — filtering irrelevant context before answering
3. **Prompt Chaining** — sequential prompts where output feeds input
4. **Meta Prompting** — using the model to optimize its own prompts

In [ ]:
# Configuration — set your provider and model here
PROVIDER = "openai"  # "openai", "anthropic", or "ollama"
MODEL = None          # None = provider default, or specify e.g. "gpt-4o", "claude-sonnet-4-20250514"

# Meta-prompting settings
META_ITERATIONS = 3   # Number of optimization rounds

In [ ]:
import sys
sys.path.insert(0, "..")

from utils.llm_client import call_llm
import re
import time
import json

---
## 1. Chain of Draft (CoD) vs. Chain-of-Thought (CoT)

The same math problem, solved two ways: verbose CoT reasoning vs. concise CoD drafts.
Compare the token usage and accuracy.

In [ ]:
cod_problem = (
    "A store has 5 boxes of apples with 24 apples each. "
    "They sell 18 apples. How many apples are left?"
)

# Chain-of-Thought — verbose step-by-step
cot_prompt = f"Q: {cod_problem}\nA: Let's think step by step."

# Chain of Draft — concise draft steps (5 words max per step)
cod_prompt = (
    f"Q: {cod_problem}\n"
    f"A: [draft]: concise reasoning steps, 5 words max each."
)

print("=" * 60)
print("CHAIN-OF-THOUGHT (verbose)")
print("=" * 60)
cot_start = time.time()
cot_response = call_llm(cot_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
cot_elapsed = time.time() - cot_start
print(cot_response)
print(f"\nLatency: {cot_elapsed:.2f}s")

print()
print("=" * 60)
print("CHAIN OF DRAFT (concise)")
print("=" * 60)
cod_start = time.time()
cod_response = call_llm(cod_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
cod_elapsed = time.time() - cod_start
print(cod_response)
print(f"\nLatency: {cod_elapsed:.2f}s")

### Analysis

CoD produces the same correct answer with dramatically fewer tokens. The reasoning steps
are compressed into concise drafts. At production scale, this token savings translates
directly to cost and latency reduction.

In [ ]:
# Token comparison (approximate — word count as proxy)
cot_words = len(cot_response.split())
cod_words = len(cod_response.split())
reduction = (1 - cod_words / cot_words) * 100 if cot_words > 0 else 0

print(f"CoT word count: ~{cot_words}")
print(f"CoD word count: ~{cod_words}")
print(f"Approximate reduction: {reduction:.0f}%")
print(f"\nNote: Actual token savings are typically larger because CoD")
print(f"avoids verbose connectors and explanations.")

---
## 2. System 2 Attention (S2A)

Test whether S2A recovers accuracy when the model is given opinionated or irrelevant context.
We compare: direct answer with noisy context vs. S2A-filtered answer.

In [ ]:
s2a_question = "When was the Eiffel Tower built?"

opinionated_context = (
    "I think the Eiffel Tower is the most overrated landmark in Europe. "
    "The Colosseum in Rome is far more historically significant. "
    "Most tourists just go to Paris because it's trendy, not because "
    "the Eiffel Tower is actually worth seeing. "
    f"Question: {s2a_question}"
)

# Direct answer — no S2A filtering
direct_prompt = f"Context: {opinionated_context}\nAnswer the question based on the context."

# S2A Step 1: Filter irrelevant/opinionated context
s2a_filter_prompt = (
    "Given the following text by a user, extract the part that is "
    "unbiased and not their opinion, so that using that text alone "
    "would be good context for providing an unbiased answer to the "
    "question portion of the text.\n\n"
    "Please include the actual question or query that the user is asking. "
    "Label with 'Unbiased text context:' and 'Question/Query:'.\n\n"
    f"Text by User: {opinionated_context}"
)

print("=" * 60)
print("DIRECT ANSWER (with opinionated context)")
print("=" * 60)
direct_response = call_llm(direct_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(direct_response)

print()
print("=" * 60)
print("S2A STEP 1: Context Filtering")
print("=" * 60)
filtered_context = call_llm(s2a_filter_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(filtered_context)

print()
print("=" * 60)
print("S2A STEP 2: Answer with filtered context")
print("=" * 60)
s2a_answer_prompt = f"{filtered_context}\nAnswer the question."
s2a_response = call_llm(s2a_answer_prompt, provider=PROVIDER, model=MODEL, temperature=0.0)
print(s2a_response)

### Analysis

The direct answer may absorb the opinionated context and produce a hedged, sycophantic response.
S2A filters the opinion out first, producing a clean factual answer. The extra LLM call is
worth it when objectivity and factual accuracy matter.

**2026 caveat**: Frontier models (GPT-4o, Claude 3.5) are better at ignoring noise than
the LLaMA-2-70B tested in the original paper. Test whether your model actually degrades
before adding S2A overhead.

---
## 3. Prompt Chaining

Demonstrate a 3-step chain: Summarize → Classify → Extract structured fields.
Each prompt is simpler and more reliable than doing all three in one prompt.

In [ ]:
# Step 1: Summarize the document
def chain_step_summarize(document: str) -> str:
    """Summarize a document in 2-3 sentences."""
    prompt = (
        f"Summarize the following document in 2-3 concise sentences.\n\n"
        f"Document:\n{document}\n\n"
        f"Summary:"
    )
    return call_llm(prompt, provider=PROVIDER, model=MODEL, temperature=0.0)


# Step 2: Classify the summary
def chain_step_classify(summary: str) -> str:
    """Classify a summary into one of predefined categories."""
    prompt = (
        f"Classify the following summary into exactly one category: "
        f"[TECHNOLOGY, BUSINESS, HEALTH, EDUCATION, OTHER].\n\n"
        f"Summary: {summary}\n\n"
        f"Category:"
    )
    return call_llm(prompt, provider=PROVIDER, model=MODEL, temperature=0.0)


# Step 3: Extract structured fields
def chain_step_extract(summary: str, category: str) -> str:
    """Extract structured fields from a summary."""
    prompt = (
        f"Extract structured fields from the following summary. "
        f"Return valid JSON with keys: title, key_topic, category, "
        f"sentiment (positive/negative/neutral), action_items (list).\n\n"
        f"Category: {category}\n"
        f"Summary: {summary}\n\n"
        f"JSON:"
    )
    return call_llm(prompt, provider=PROVIDER, model=MODEL, temperature=0.0)


# Run the full chain
sample_document = (
    "Apple announced today that it will invest $500 million in a new AI research "
    "center in Austin, Texas. The facility will focus on developing next-generation "
    "language models and will create 200 new engineering jobs over the next two years. "
    "CEO Tim Cook stated that this investment reflects Apple's commitment to "
    "advancing artificial intelligence while maintaining its focus on user privacy. "
    "The center is expected to open in Q3 2027."
)

print("INPUT DOCUMENT:")
print(sample_document)
print()

# Step 1
print("=" * 60)
print("STEP 1: SUMMARIZE")
print("=" * 60)
start = time.time()
summary = chain_step_summarize(sample_document)
step1_time = time.time() - start
print(summary)
print(f"Latency: {step1_time:.2f}s")

# Step 2
print()
print("=" * 60)
print("STEP 2: CLASSIFY")
print("=" * 60)
start = time.time()
category = chain_step_classify(summary)
step2_time = time.time() - start
print(category)
print(f"Latency: {step2_time:.2f}s")

# Step 3
print()
print("=" * 60)
print("STEP 3: EXTRACT STRUCTURED FIELDS")
print("=" * 60)
start = time.time()
extracted = chain_step_extract(summary, category)
step3_time = time.time() - start
print(extracted)
print(f"Latency: {step3_time:.2f}s")

# Total chain metrics
print()
print("=" * 60)
print("CHAIN METRICS")
print("=" * 60)
print(f"Total chain latency: {step1_time + step2_time + step3_time:.2f}s")
print(f"Steps: 3 | Each step is simpler than a single combined prompt")
print(f"Risk: Error propagation — if Step 1 summary is wrong, Steps 2-3 amplify the error")

### Analysis

Each step in the chain is focused and constrained — easier to validate and debug than a
single prompt that tries to summarize, classify, and extract simultaneously. The tradeoff
is latency accumulation and error propagation. In production, add validation between steps
(e.g., check that the category is in the allowed set, check that the JSON is valid).

---
## 4. Meta Prompting

Use the model to evaluate and improve its own prompt through iterative refinement.
We start with a mediocre prompt and let the model optimize it.

In [ ]:
def meta_evaluate_prompt(task: str, prompt: str, provider: str, model) -> dict:
    """Evaluate a prompt's quality using the LLM as judge."""
    eval_prompt = (
        f"You are a prompt quality evaluator. Given a task and a prompt, "
        f"rate the prompt on a scale of 1-10 for: clarity, specificity, "
        f"and likelihood of producing correct output.\n\n"
        f"Task: {task}\n"
        f"Prompt: {prompt}\n\n"
        f"Respond with JSON: {{\"clarity\": N, \"specificity\": N, "
        f"\"correctness\": N, \"feedback\": \"...\"}}"
    )
    response = call_llm(eval_prompt, provider=provider, model=model, temperature=0.0)
    try:
        # Extract JSON from response
        json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except json.JSONDecodeError:
        pass
    return {"clarity": 5, "specificity": 5, "correctness": 5, "feedback": response}


def meta_optimize_prompt(task: str, current_prompt: str, eval_result: dict,
                         provider: str, model) -> str:
    """Generate an improved version of the prompt based on evaluation feedback."""
    optimize_prompt = (
        f"You are a prompt engineer. Improve the following prompt based on "
        f"the evaluation feedback. The task is: {task}\n\n"
        f"Current prompt: {current_prompt}\n\n"
        f"Evaluation feedback: {eval_result.get('feedback', 'No feedback')}\n"
        f"Scores — clarity: {eval_result.get('clarity', '?')}/10, "
        f"specificity: {eval_result.get('specificity', '?')}/10, "
        f"correctness: {eval_result.get('correctness', '?')}/10\n\n"
        f"Return ONLY the improved prompt, nothing else."
    )
    return call_llm(optimize_prompt, provider=provider, model=model, temperature=0.3)


# Define the task and initial (mediocre) prompt
meta_task = "Classify a customer support email as: URGENT, BILLING, TECHNICAL, or GENERAL"
initial_prompt = "Read this email and tell me what it's about."

print("=" * 60)
print("META PROMPTING OPTIMIZATION LOOP")
print("=" * 60)
print(f"Task: {meta_task}")
print(f"Initial prompt: {initial_prompt}")
print()

current_prompt = initial_prompt
for i in range(META_ITERATIONS):
    print(f"--- Iteration {i + 1} ---")
    
    # Evaluate
    eval_result = meta_evaluate_prompt(meta_task, current_prompt, PROVIDER, MODEL)
    print(f"Scores: clarity={eval_result.get('clarity', '?')}, "
          f"specificity={eval_result.get('specificity', '?')}, "
          f"correctness={eval_result.get('correctness', '?')}")
    print(f"Feedback: {eval_result.get('feedback', 'N/A')}")
    
    # Optimize
    current_prompt = meta_optimize_prompt(meta_task, current_prompt, eval_result, PROVIDER, MODEL)
    print(f"Optimized prompt: {current_prompt}")
    print()

print("=" * 60)
print("FINAL OPTIMIZED PROMPT")
print("=" * 60)
print(current_prompt)

In [ ]:
# Test the initial vs optimized prompt on a sample email
sample_email = (
    "Hi, I was charged twice for my subscription this month. "
    "My card ending in 4521 shows two charges of $29.99 on July 15th. "
    "Please refund the extra charge."
)

print("=" * 60)
print("TEST: Initial prompt on sample email")
print("=" * 60)
initial_result = call_llm(
    f"{initial_prompt}\n\nEmail: {sample_email}",
    provider=PROVIDER, model=MODEL, temperature=0.0
)
print(initial_result)

print()
print("=" * 60)
print("TEST: Optimized prompt on sample email")
print("=" * 60)
optimized_result = call_llm(
    f"{current_prompt}\n\nEmail: {sample_email}",
    provider=PROVIDER, model=MODEL, temperature=0.0
)
print(optimized_result)

### Analysis

The initial prompt ("Read this email and tell me what it's about") is vague — it does not
specify the output format or the categories. After meta-prompting optimization, the prompt
should include explicit category labels, output format constraints, and classification criteria.

**Critical guardrail**: In production, always evaluate optimized prompts on a **held-out**
test set, not the same examples used during optimization. Without holdout evaluation,
the optimized prompt may memorize rather than generalize.

---
## Summary

| Technique | What it does | Cost overhead | When to use |
|---|---|---|---|
| Chain of Draft | Concise reasoning steps (~5 words each) | **Saves** ~80% tokens | High-volume production, cost-sensitive paths |
| System 2 Attention | Filters irrelevant context before answering | 2x LLM calls | Opinionated/noisy inputs, accuracy-critical |
| Prompt Chaining | Sequential prompts, output feeds input | N× latency (N steps) | Multi-stage tasks too complex for one prompt |
| Meta Prompting | Model generates and optimizes its own prompts | Multiple eval+optimize cycles | Scaling prompt engineering across many tasks |

**Key insight**: These techniques are composable. You can chain CoD prompts together,
apply S2A within a chain, or use meta-prompting to optimize individual links in a chain.
The right combination depends on your model, task, and production constraints.